# Chọn rerank_k cho pilot — chỉ dùng DEV

So sánh `30/20/10` bằng đúng e5-task2 + ce-task + blend 30% trên bốn lát dev: paraphrase/title × có dấu/không dấu. Notebook chỉ chạy **DEV**, không đọc test và không tự đổi cấu hình pilot.

**Trước khi chạy:** Runtime → Change runtime type → T4 GPU. Dữ liệu Jira đã scrub nhưng vẫn chứa tên service, mã lỗi và luồng nghiệp vụ; kiểm tra quy định đơn vị trước khi upload.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "sentence-transformers==6.0.1" gdown

In [ ]:
from google.colab import files
up = files.upload()  # chọn rerank_k_bundle.zip
BUNDLE = next(iter(up))
!unzip -oq {BUNDLE} -d /content/work
%cd /content/work
!ls pipeline data/processed

## Tải đúng hai checkpoint pilot

In [ ]:
import os, glob, shutil, zipfile, gdown
FILES = {"e5-base-task2": "1rNO17nLa1kXGdmMEsUhW5Hl_9e7kjN8O",
         "bge-reranker-base-task": "12x-H0yLkedkl0_94K_CHV621cZuncRqn"}
os.makedirs("models", exist_ok=True)
for name, fid in FILES.items():
    dst, z = "models/" + name, "/content/" + name + ".zip"
    if not os.path.exists(dst + "/config.json"):
        gdown.download(id=fid, output=z, quiet=False)
        assert zipfile.is_zipfile(z), "Không tải được model ZIP: " + name
        tmp = "/content/unz_" + name
        shutil.rmtree(tmp, ignore_errors=True)
        with zipfile.ZipFile(z) as archive: archive.extractall(tmp)
        cfg = sorted(glob.glob(tmp + "/**/config.json", recursive=True), key=len)
        shutil.move(os.path.dirname(cfg[0]), dst)
    print(name, os.path.exists(dst + "/config.json"))

## Smoke test 2 query

In [ ]:
!python pipeline/eval_rerank_k.py --device cuda --limit 2 --latency-queries 0 --n-boot 100 --ce-batch 128 --score-chunk 50

## Chạy toàn bộ DEV

Latency GPU không đại diện máy pilot nên để `0`. Sau khi tải kết quả về, chạy lại script local với cache hoàn chỉnh để đo CPU.

In [ ]:
!python pipeline/eval_rerank_k.py --device cuda --latency-queries 0 --n-boot 5000 --ce-batch 128 --score-chunk 50

## Tải kết quả về

Giải nén ZIP vào thư mục dự án, sau đó chạy local:
`python pipeline/eval_rerank_k.py --latency-queries 10`
để bổ sung latency CPU mà không chấm CE lại.

In [ ]:
!zip -q rerank_k_results.zip data/processed/rerank_k_dev_scores.jsonl data/processed/rerank_k_dev.json
!ls -lh rerank_k_results.zip
files.download("rerank_k_results.zip")